# 02 — Results Comparison

This notebook aggregates the outputs of all completed experiments into report-ready tables and figures. It compares the main models, summarizes the controlled ablation studies, and examines learning dynamics and train-vs-validation gaps to support the final discussion in the report.

In [ ]:
import json
import glob
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from IPython.display import Image, display

sns.set_theme(style="whitegrid")
exp_dir = Path("experiments")
fig_dir = Path("report/figs")
table_dir = Path("report/tables")
fig_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def _flatten_metrics(record: dict) -> dict:
    config = record.get('config_snapshot', {}) or {}
    model = config.get('model', {}) or {}
    best = record.get('best', {}) or {}
    test = record.get('test', {}) or {}
    per_class = test.get('per_class', {}) or {}
    neg = per_class.get('neg', {}) or {}
    pos = per_class.get('pos', {}) or {}
    return {
        'experiment': record.get('experiment_name'),
        'model_type': model.get('type'),
        'embed_dim': model.get('embed_dim'),
        'hidden_dim': model.get('hidden_dim'),
        'dropout': model.get('dropout'),
        'num_layers': model.get('num_layers'),
        'best_epoch': best.get('epoch'),
        'val_accuracy': best.get('val_accuracy'),
        'val_f1': best.get('val_f1'),
        'test_acc': test.get('accuracy'),
        'test_f1': test.get('f1'),
        'test_precision': test.get('precision'),
        'test_recall': test.get('recall'),
        'test_pos_f1': pos.get('f1'),
        'test_neg_f1': neg.get('f1'),
        'num_params': record.get('num_params'),
        'wallclock_seconds': record.get('wallclock_seconds'),
    }

records = []
for path in glob.glob(str(exp_dir / '*' / 'metrics.json')):
    with open(path, 'r', encoding='utf-8') as f:
        records.append(_flatten_metrics(json.load(f)))

results_df = pd.DataFrame(records)
results_df = results_df.sort_values(['model_type', 'experiment']).reset_index(drop=True)
results_df

In [ ]:
main_models = results_df[results_df['experiment'].isin(['mlp_main', 'lstm_main', 'bilstm_attn_main'])].copy()
main_models = main_models.sort_values('test_f1', ascending=False)
main_table = main_models[[
    'experiment', 'model_type', 'embed_dim', 'hidden_dim', 'dropout', 'num_layers',
    'best_epoch', 'val_accuracy', 'val_f1', 'test_acc', 'test_f1', 'test_precision', 'test_recall',
    'test_pos_f1', 'test_neg_f1', 'num_params', 'wallclock_seconds'
]].copy()
main_md = main_table.to_markdown(index=False)
(table_dir / 'main_comparison.md').write_text(main_md, encoding='utf-8')
main_table

In [ ]:
embed_df = results_df[results_df['experiment'].str.startswith('exp_embed_dim_')].copy()
embed_df = embed_df.sort_values('embed_dim')
embed_table = embed_df[['embed_dim', 'val_f1', 'test_f1', 'num_params']].copy()
embed_md = embed_table.to_markdown(index=False)
(table_dir / 'exp_embed_dim.md').write_text(embed_md, encoding='utf-8')
embed_table

In [ ]:
dropout_df = results_df[results_df['experiment'].str.startswith('exp_dropout_')].copy()
dropout_df = dropout_df.sort_values('dropout')
dropout_table = dropout_df[['dropout', 'val_f1', 'test_f1']].copy()
dropout_md = dropout_table.to_markdown(index=False)
(table_dir / 'exp_dropout.md').write_text(dropout_md, encoding='utf-8')
dropout_table

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=False)
for ax, exp_name in zip(axes, ['mlp_main', 'lstm_main', 'bilstm_attn_main']):
    metrics_path = exp_dir / exp_name / 'metrics.csv'
    df = pd.read_csv(metrics_path)
    ax.plot(df['epoch'], df['train_loss'], marker='o', linewidth=2, markersize=4, label='Train loss')
    ax.plot(df['epoch'], df['val_loss'], marker='s', linewidth=2, markersize=4, label='Val loss')
    ax.set_title(exp_name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('Learning curves for main models')
fig.tight_layout()
fig.savefig(fig_dir / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
subprocess.run([
    'python', 'scripts/plot_ablation.py',
    '--experiments', 'exp_embed_dim_64', 'exp_embed_dim_128', 'exp_embed_dim_256',
    '--labels', 'emb_dim=64', 'emb_dim=128', 'emb_dim=256',
    '--title', 'Experiment 1: Embedding Dimension Ablation (Training Curves)',
    '--output', 'report/figs/ablation_embed_dim.png'
], check=True)
subprocess.run([
    'python', 'scripts/plot_ablation.py',
    '--experiments', 'exp_dropout_0.2', 'exp_dropout_0.3', 'exp_dropout_0.5',
    '--labels', 'dropout=0.2', 'dropout=0.3', 'dropout=0.5',
    '--title', 'Experiment 2: Dropout Ablation (Training Curves)',
    '--output', 'report/figs/ablation_dropout.png'
], check=True)

display(Image(filename=str(fig_dir / 'ablation_embed_dim.png')))
display(Image(filename=str(fig_dir / 'ablation_dropout.png')))

In [ ]:
def _verdict(val_acc: float, gap: float) -> str:
    if val_acc < 0.7:
        return 'underfit'
    if gap < 0.02:
        return 'fit'
    if gap <= 0.05:
        return 'mild overfit'
    return 'strong overfit'

gap_rows = []
for exp_name in ['mlp_main', 'lstm_main', 'bilstm_attn_main']:
    df = pd.read_csv(exp_dir / exp_name / 'metrics.csv')
    best_idx = df['val_f1'].idxmax()
    row = df.loc[best_idx]
    gap = float(row['train_acc'] - row['val_accuracy'])
    gap_rows.append({
        'experiment': exp_name,
        'best_epoch': int(row['epoch']),
        'train_acc': float(row['train_acc']),
        'val_accuracy': float(row['val_accuracy']),
        'gap': gap,
        'verdict': _verdict(float(row['val_accuracy']), gap),
    })

gap_df = pd.DataFrame(gap_rows)
print(gap_df.to_string(index=False, formatters={'train_acc': '{:.4f}'.format, 'val_accuracy': '{:.4f}'.format, 'gap': '{:.4f}'.format}))
gap_df

## Summary

- The best overall model is the one with the highest test F1 among the three main runs.
- Increasing embedding dimension should improve performance up to a point, but larger embeddings also increase the parameter count.
- Dropout changes the balance between fitting and generalization, with intermediate values often performing best.
- The learning-curve and gap analysis help identify whether the main models are underfitting or overfitting.